In [ ]:
import numpy as np
from datetime import datetime

def source_activity(file_date, source):
    '''
    Function that calculates the remaining activity of a radioisotope source 
    by applying its halflife and total time to its initial activity. 

    is dependant on the source type where the function chooses
    which hardcoded source values to use.
    __________________
    Parameters
        init_activity: Float
        The initial activity of the radio-isotope from when it was first calibrated.
        
    ___________________
    Outputs
        
    '''
    #make this into a dictinary where the source is the key
    source_params = {
        "sources": ['Am','Ba','Co','Cs'],
        "initial_activity": [11.92, 10.82, 11.32, 12.41],
        "accuracy": [5.0, 4.8, 1.9, 3.7],
        "HL3_confirmed": [432.2, 10.537, 5.2714, 30.09],
        "emission_frac": [0.999,0.999,0.999,0.999]
    }

    # index corresponding to source
    i = source_params["sources"].index(source)
    
    # getting corresponding values within dictionary 
    init_activity = source_params["initial_activity"][i]
    accuracy = source_params["accuracy"][i]
    half_life = source_params["HL3_confirmed"][i]
    emission_frac = source_params["emission_frac"][i]
    
    # converting half_life to seconds considering leap years
    HL_seconds = half_life * 365.25 * 24 * 60 * 60 
    
    time = source_age(file_date)
    remaining_activity = init_activity * (0.5 **(time/HL_seconds))
    activity = remaining_activity * emission_frac

    return activity



def eff_info_spe(filename):
    """
    Function that reads an .Spe file and returns date measured and the type of source.
    
    **NOTE** There is no information in the files that states what radio-isotope
    the file belongs to, other than the file naming convention used when
    collecting data. For this reason this function relys on that fact when returning
    the radio-isotope.
    """
    source = str(filename).split('_')[1]
    
    with open(filename, "r") as f:
        lines = [line.strip() for line in f]

    # $DATE_MEA: is followed by a line like "10/12/2026 16:18:58" 
    meas_date = lines[lines.index('$DATE_MEA:') + 1]

    return meas_date, source



def eff_info_mca(filename):
    """
    Function that reads an .mca file and returns date measured and the type of source.
    
    **NOTE** There is no information in the files that states what radio-isotope
    the file belongs to, other than the file naming convention used when
    collecting data. For this reason this function relys on that fact when returning
    the radio-isotope.
    """
    source = str(filename).split('_')[1]

    with open(filename, 'r') as file:
        spectrum_data = file.readlines()

    meas_date = spectrum_data[9].replace('START_TIME - ', '').strip()
    
    return meas_date, source



def extract_eff_info(filename):
    """
    Function that reads any supported spectrum file and returns start time
    and source type throughthe functions it calls.
    """
    cutis_folder = Path(__file__).parent / "CUITS"
   
    # Create the complete path to the specific file
    full_path = cutis_folder / filename
    # Extract the file extension (e.g., '.mca')
    suffix = full_path.suffix.lower()
    #suffix = Path(filename).suffix.lower()
    if suffix == ".spe":
        return eff_info_spe(full_path)
    if suffix == ".mca":
        return eff_info_mca(full_path)
    
def source_age(meas_date):
    '''
    Function that calculates the age of the source from the time that the
    source was calibrated and the time it was measured.
    '''
    
    # all measurements taken from upstairs sources.  
    calibration_date = '1979/02/01 12:00:00'

    cal_time = datetime.strptime(calibration_date, "%Y/%m/%d %H:%M:%S")
    meas_time = datetime.strptime(meas_date, "%m/%d/%Y %H:%M:%S")

    age_of_source = (meas_time - cal_time).total_seconds()
    
    return age_of_source


In [ ]:
def abs_efficiency(filename):
    '''
    Function that calculates and returns the the absolute efficiency
    of a detector at corresponding energy levels of a radioisotope.
    This efficiency is found as the ratio between the decays that the detector
    picks up and the total decays from the source over a unit time.
    __________________
    Parameters
        filename: file path
        
    ___________________
    Outputs
        absolute_efficiency: list of floats?
        The absolute efficiency/ efficiencies of the detector at a given energy level.
    '''
    # just assume this works
    peaks = co.pick_peaks(
    windows=[
        (506, 693, [1173.228, 1332.492], [547, 632]) # delete all this so that it works via the interactive selector
    ])
    
    # Emission fractions for Cobalt from the lab manual. need to add more for the rest of the elements
    emission_fractions = {
        1173.228: 0.9985,
        1332.492: 0.999826
    }

    #expected values
    # "Cs": [661.657],
    # "Co": [1173.228, 1332.492],
    # "Ba": [302.8508, 356.0129],
    # "Am": [26.3446]

    file_date, source = extract_eff_info(filename)
    activity_uCi = source_activity(file_date, source)

    # Convert microcuries to Becquerels
    activity_bq = activity_uCi * 37000

    for peak in peaks:
        energy = peak["energy"]
        count_rate = peak["area"]

        emission_fraction = emission_fractions[energy]

        absolute_efficiency = count_rate / (activity_bq * emission_fraction)

        print(
            f"Energy: {energy:.3f} keV, "
            f"Rate: {count_rate:.3f} cps, "
            f"Efficiency: {absolute_efficiency:.5f}"
        )

    return absolute_efficiency

In [1]:
def intrinsic_eff(asb_effic, filename):
    '''
    Function that calculates the intrinsic efficiency of the detector.
    This is done by determining and applying a geometry factor which 
    itself is dependant on the detectors radius and distance from 
    the radio isotope source.
    __________________
    Parameters
        dist: Float
        The distance the detector is from the radioisotope

        radius: Float
        radius of the detector

        abs_effic: Float
        absolute efficiency of the detector
    ___________________
    Outputs
        intrinsic_efficiency: Float
        intrinsic efficiency of the detector
    
    '''

    #get detector from file name
    detector = str(filename).split('_')[0]

    # radius list of detectors
    dist = 0.1 # meters
    rad_list =  {
        "BGO": #radius of bgo detector,
        "NaI": #radius of nai detector,
        "CdTe": #radius of bcdte detector
    }

    radius = rad_list[detector]

    #geometry factor
    G = 0.5 * (1 - (dist / (sqrt((dist**2)+(radius**2))))) 
    intrinsic_efficiency = abs_effic / G
    
    return intrinsic_efficiency

SyntaxError: invalid syntax (2382680149.py, line 31)